   Remark: The widgets used to set and cancel targets were working properly before the implementation of the dynamic graphs. After noticing that sometimes, the success of reaching a target is detected by the graphs, but not by the buttons, whereas the two are modified in the same section of code, I considered that it might be a consequence of the fact that my processor is used at 100% when running all the cells.

## Source

In [1]:
import sys
sys.path.append('/root/ros_ws/devel/lib/python3/dist-packages')
sys.path.append('/opt/ros/noetic/lib/python3/dist-packages')

## Imports

In [2]:
import rospy
import actionlib
import threading
import time
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import clear_output
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
from assignment_2_2024.msg import RobotOdom, PlanningAction, PlanningGoal
from actionlib_msgs.msg import GoalStatus

## Generate random targets

In [3]:
import random
import pandas as pd

name_file='random_targets_20.csv'
nb_randoms = 20
def generate_random_targets_csv(n, name_file):
    data = {
        'x': [random.randint(-9, 9) for _ in range(n)],
        'y': [random.randint(-9, 9) for _ in range(n)]
    }
    df = pd.DataFrame(data)
    df.to_csv(name_file, index=False)
    print(f"{n} random targets saved in '{name_file}'")

#generate_random_targets_csv(nb_randoms, name_file)

In [4]:
read_file='random_targets_20.csv'
def read_targets(read_file):
    df = pd.read_csv(read_file)
    
    random_targets = [
        df['x'].tolist(),
        df['y'].tolist()
    ]
    
    return random_targets

# Exemple d'utilisation
random_targets = read_targets(read_file)

## Set or cancel a target

In [5]:
nb_success, nb_targets = 0, 0
time_to_reach = []
def set_target_client():
    global nb_success, nb_targets
    client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
    
    connected = client.wait_for_server(timeout=rospy.Duration(5.0))
    if not connected:
        rospy.logerr("Could not connect to /reaching_goal action server.")
    else:
        rospy.loginfo("Connected to /reaching_goal action server.")

    for x_rd, y_rd in zip(random_targets[0], random_targets[1]):
        nb_targets += 1
        print(f"Target sent({nb_targets}/{nb_randoms}): x={x_rd}, y={y_rd}")
        
        rospy.set_param('/des_pos_x', x_rd)
        rospy.set_param('/des_pos_y', y_rd)
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x_rd
        goal.target_pose.pose.position.y = y_rd
        
        start_time = time.time()
        client.send_goal(goal)
        print("waiting for result")
        client.wait_for_result()
        end_time = time.time()
        
        duration = end_time - start_time
        
        state = client.get_state()
        if state == GoalStatus.SUCCEEDED:
            nb_success += 1
            rospy.loginfo(f"Target reached: ({x_rd},{y_rd})")
            time_to_reach.append(duration)
            rospy.sleep(1.0)
        else:
            rospy.logwarn(f"Failed to reach target: ({x_rd},{y_rd}) (client state: {state})")
            time_to_reach.append("fail")

    print("All targets reached")

## Position and distance to the obstacles feedback

In [6]:
x_rob, y_rob = 0, 0
def get_feedback():
    # Widgets
    x_widget = widgets.FloatText(description='X:', disabled=True)
    y_widget = widgets.FloatText(description='Y:', disabled=True)
    min_dist_widget = widgets.FloatText(description='Obstacle [m]:', disabled=True)
    pos_box = widgets.VBox([widgets.Label("Robot Position:"), x_widget, y_widget])
    laser_box = widgets.VBox([widgets.Label("Closest Obstacle:"), min_dist_widget])
    main_box = widgets.HBox([pos_box, laser_box])
    display(main_box)

    # Callbacks
    def pos_callback(msg):
        global x_rob, y_rob
        x_widget.value = msg.pose.pose.position.x
        y_widget.value = msg.pose.pose.position.y
        x_rob = x_widget.value
        y_rob = y_widget.value

    def laser_callback(msg):
        min_dist_widget.value = min(msg.ranges)

    # Subscribers /odom, /scan
    rospy.Subscriber('/odom', Odometry, pos_callback)
    rospy.Subscriber('/scan', LaserScan, laser_callback)

## Display the interface

In [7]:
if not rospy.core.is_initialized():
    rospy.init_node('action_client', anonymous=True)

In [8]:
get_feedback()

In [9]:
set_target_client()

Target sent(1/20): x=6, y=1
waiting for result
Target sent(2/20): x=-6, y=-2
waiting for result


KeyboardInterrupt: 

## Position of the robot

In [ ]:
#%matplotlib notebook
#fig, ax = plt.subplots() 
#xdata, ydata = [], []
#ax.set_xlim(-10,10)
#ax.set_ylim(-10,10)
#ln, = plt.plot([], [], 'r-')

#def init():
#    ln.set_data([],[])
#    return ln,

#def update(frame): 
 #   xdata.append(x_rob) 
  #  ydata.append(y_rob) 
   # ln.set_data(xdata, ydata) 
    #return ln,

#ani = FuncAnimation(fig, update, frames=range(50), init_func=init, blit=True)
#plt.show()

## Number of targets reached/not reached

In [10]:
#%matplotlib notebook
#fig2, ax2 = plt.subplots()
#bars = ax2.bar(['Success', 'Failure'], [nb_success, nb_targets-nb_success], color=['green', 'red'])
#ax2.set_ylim(0,20)

#def init2():
 #   bars[0].set_height(nb_success)
  #  bars[1].set_height(nb_targets-nb_success)
   # return bars
#def update2(frame):
 #   global nb_success, nb_targets
  #  bars[0].set_height(nb_success)
   # bars[1].set_height(nb_targets-nb_success)
    #return bars
#ani2 = FuncAnimation(fig2, update2, frames=range(100), init_func=init2, blit=True, interval=500)

In [10]:
file_out='duration_right_25.csv'
df = pd.DataFrame({
        'x': random_targets[0],
        'y': random_targets[1],
        'duration_sec': time_to_reach
    })
df.to_csv(file_out, index=False)
print(f"Durations to reach targets saved in {file_out}.")

Durations to reach targets saved in duration_right.csv.


In [ ]:
print(time_to_reach)